# 03 - Validação cruzada (Parte 11)

Escolhe o modelo por **previsão fora da amostra**.
Todo modelo usa só covariáveis anteriores ao 2º turno e, no município de teste, o peso n é o n̂ estimado (a UF retida ainda não foi apurada). Dentro da amostra, um binomial com ligação canônica e dummies de região reproduz os totais regionais exatamente (notebook 02, célula 3), então a
estimativa regional só tem conteúdo se vier de um modelo que não viu os dados que prevê (a estimativa regional do modelo final está no notebook 05).

## Parte 11 - Validação cruzada

**Regra de decisão (fixada antes de ver os resultados).**
1. *Métrica primária*: erro absoluto médio do share de cada UF, em p.p., na validação *leave-one-UF-out* (LOSO), média sobre as 27 UFs. Imita prever um estado
   inédito, e o alvo final são shares regionais (soma de UFs), não municípios. O share da UF é ponderado por votos.
2. *Regra de 1 erro-padrão*: o erro-padrão (EP) é o desvio dos 27 erros por UF do melhor modelo dividido por √27. Todo modelo com erro médio até
   (melhor + EP) é considerado empatado; entre os empatados fica **o de menos parâmetros** (parcimônia).
3. *Métrica secundária*: MAE municipal ponderado por votos na validação 5-fold (semente 0), usada para desempatar quando há igualdade de parâmetros e como verificação.
4. LOSO é a validação honesta porque a autocorrelação espacial é forte (I de Moran do share 2026 = 0,896): em 5-fold aleatório, os vizinhos de um município de teste
   estão no treino e o erro sai otimista.
5. Modelos com efeito fixo de UF (`quasi_uf`, `bb_uf`) só entram no 5-fold: a UF retida não tem efeito próprio no treino. `quasi_logit` dá o mesmo π̂ de `bin_logit`
   (mesmo estimador de β; muda só a variância) e é avaliado uma vez, como `bin_logit`.

In [1]:
import sys, json, warnings, time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
from multiprocessing import get_context
import numpy as np, pandas as pd
import plotly.graph_objects as go
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import especificacoes as E, modelos as M
pd.options.display.float_format = "{:,.4f}".format
T0 = time.time()
RECALCULAR = True   # True: refaz a validação; False: usa results/cv/*.parquet quando existir
TAB, FIG, CV = ROOT / "results/tables", ROOT / "results/figures", ROOT / "results/cv"
for p in (TAB, FIG, CV): p.mkdir(parents=True, exist_ok=True)
df = pd.read_parquet(ROOT / "data/processed/municipios.parquet").reset_index(drop=True)
mapa = E.MAPA_2026
sel = json.loads((TAB / "selecao_variaveis.json").read_text())
E.registrar_selecao(sel["variaveis"], regiao_fundir=tuple(tuple(p) for p in sel["regiao_fundir"]))
E.definir_dados_cv(df, mapa)
y, n = df[mapa["y"]].to_numpy(float), df[mapa["n"]].to_numpy(float)
nh = E.n_loso(df)   # n̂: peso das previsões fora da amostra (n real só no lado observado)
uf, reg = df[mapa["uf"]].astype(str).to_numpy(), df[mapa["regiao"]].astype(str).to_numpy()
UFS_ = sorted(set(uf)); REGS = sorted(set(reg))
print(len(df), "municípios modelados;", f"{y.sum()/n.sum():.4%} de votos em Lula;", len(UFS_), "UFs;", len(REGS), "regiões")
NOMES = [k for k in E.CANDIDATOS if k != "quasi_logit"]
NOMES_LOSO = [k for k in NOMES if k not in ("quasi_uf", "bb_uf")]
print(len(NOMES), "candidatos distintos;", len(NOMES_LOSO), "em LOSO")

5570 municípios modelados; 51.7796% de votos em Lula; 27 UFs; 5 regiões
20 candidatos distintos; 18 em LOSO


**Município fora do modelo.** Os totais oficiais (`resultado_regional.csv`, `resultado_uf.csv`) incluem Boa Esperança do Norte (MT), sem voto de 2022 e portanto sem a defasagem exigida (945 votos de Lula e 2.353 de Flávio em 2026).
Os 5.570 municípios acima são os modelados; **todas as comparações com o "real" usam os totais recalculados sobre essas linhas** (o efeito de somar o município fora do modelo é medido no notebook 05).

In [2]:
folds5 = M.folds_kfold(len(df), k=5, seed=0)
foldsL = M.folds_por_grupo(uf)
def _cv(nome, esquema, folds):
    arq = CV / f"{esquema}_{nome}.parquet"
    if arq.exists() and not RECALCULAR:
        return pd.read_parquet(arq)["pi"].to_numpy()
    return None
def rodar(nomes, esquema, folds):
    pend = [k for k in nomes if _cv(k, esquema, folds) is None]
    saida = {}
    if pend:
        with ProcessPoolExecutor(max_workers=14, mp_context=get_context("fork")) as ex:
            fut = {(k, i): ex.submit(E.prever_fold, k, tr, te) for k in pend for i, (tr, te) in enumerate(folds)}
            for k in pend:
                pi = np.full(len(df), np.nan)
                for i, (tr, te) in enumerate(folds): pi[te] = fut[(k, i)].result()
                pd.DataFrame({"pi": pi}).to_parquet(CV / f"{esquema}_{k}.parquet"); saida[k] = pi
    for k in nomes:
        if k not in saida: saida[k] = _cv(k, esquema, folds)
    return saida
PI5 = rodar(NOMES, "kfold", folds5)
print("5-fold:", round(time.time() - T0), "s"); PIL = rodar(NOMES_LOSO, "loso", foldsL)
print("LOSO:", round(time.time() - T0), "s")
with ProcessPoolExecutor(max_workers=14, mp_context=get_context("fork")) as ex:
    K = dict(zip(NOMES, ex.map(E.parametros_ajuste, NOMES)))
print("NaN nas previsões:", {k: int(np.isnan(v).sum()) for d in (PI5, PIL) for k, v in d.items() if np.isnan(v).any()} or "nenhum")

/home/ribamarneto/Documentos/Previsao Eleicoes/.claude/worktrees/slot1/.venv/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1486: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  wls_results = wls_model.fit(method=wls_method2)


/home/ribamarneto/Documentos/Previsao Eleicoes/.claude/worktrees/slot1/.venv/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1486: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  wls_results = wls_model.fit(method=wls_method2)


/home/ribamarneto/Documentos/Previsao Eleicoes/.claude/worktrees/slot1/.venv/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1486: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  wls_results = wls_model.fit(method=wls_method2)


/home/ribamarneto/Documentos/Previsao Eleicoes/.claude/worktrees/slot1/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1189: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = fit_method(


5-fold: 49 s


LOSO: 259 s


NaN nas previsões: nenhum


**Métricas.** Para cada modelo, sobre as previsões fora da amostra empilhadas: tudo que `metricas` devolve (MAE e RMSE ponderados por votos, MAE simples, deviance binomial, erro nacional)
mais o erro absoluto do share de cada UF e de cada região (p.p.). A dispersão vem dos 5 folds (5-fold) e das 27 UFs (LOSO); EP = desvio / √(número de unidades).

In [3]:
obs_uf = pd.Series(y).groupby(uf).sum() / pd.Series(n).groupby(uf).sum()
obs_reg = pd.Series(y).groupby(reg).sum() / pd.Series(n).groupby(reg).sum()
def erros(pi, esq, folds):
    a_uf = (M.agregar(pi, nh, uf).drop("Brasil") - obs_uf).abs() * 100
    a_rg = (M.agregar(pi, nh, reg).drop("Brasil") - obs_reg).abs() * 100
    m = M.metricas(y, n, pi)
    if esq == "5-fold": por = [M.metricas(y[te], n[te], pi[te])["mae_pond_pp"] for _, te in folds]
    else: por = [M.metricas(y[te], n[te], pi[te])["mae_pond_pp"] for _, te in folds]
    m.update(mae_uf_pp=a_uf.mean(), se_uf_pp=a_uf.std(ddof=1) / np.sqrt(len(a_uf)),
             mae_reg_pp=a_rg.mean(), max_reg_pp=a_rg.max(),
             se_mae_pond_pp=np.std(por, ddof=1) / np.sqrt(len(por)))
    return m, a_uf, a_rg
def tabela(PI, esq, folds):
    L, UFE = [], []
    for k, pi in PI.items():
        m, a_uf, _ = erros(pi, esq, folds)
        L.append({"modelo": k, "familia": E.CANDIDATOS[k].familia, "k": K[k], **m})
        if esq == "LOSO":
            pu = M.agregar(pi, nh, uf).drop("Brasil")
            mu = pd.Series({u: M.metricas(y[uf == u], n[uf == u], pi[uf == u])["mae_pond_pp"] for u in UFS_})
            UFE.append(pd.DataFrame({"modelo": k, "uf": UFS_, "obs_pct": obs_uf[UFS_].values * 100, "prev_pct": pu[UFS_].values * 100,
                                     "erro_uf_pp": (pu[UFS_] - obs_uf[UFS_]).values * 100, "mae_pond_municipal_pp": mu[UFS_].values}))
    t = pd.DataFrame(L)
    return t, (pd.concat(UFE) if UFE else None)
T5, _ = tabela(PI5, "5-fold", folds5)
TL, TUF = tabela(PIL, "LOSO", foldsL)
T5.sort_values("mae_pond_pp").to_csv(TAB / "cv_kfold.csv", index=False)
TL.sort_values("mae_uf_pp").to_csv(TAB / "cv_loso.csv", index=False)
TUF.to_csv(TAB / "cv_uf_loso.csv", index=False)
col = ["modelo", "k", "mae_uf_pp", "se_uf_pp", "mae_reg_pp", "mae_pond_pp", "rmse_pond_pp", "deviance", "erro_nacional_pp"]
print("LOSO (ordenado pelo erro médio por UF)"); print(TL.sort_values("mae_uf_pp")[col].to_string(index=False))
print("\n5-fold (ordenado pelo MAE ponderado)"); print(T5.sort_values("mae_pond_pp")[col[:2] + ["mae_pond_pp", "se_mae_pond_pp", "mae_uf_pp", "mae_reg_pp", "rmse_pond_pp", "deviance", "erro_nacional_pp"]].to_string(index=False))

LOSO (ordenado pelo erro médio por UF)
      modelo  k  mae_uf_pp  se_uf_pp  mae_reg_pp  mae_pond_pp  rmse_pond_pp     deviance  erro_nacional_pp
  bin_probit 25     0.9984    0.1347      0.0967       1.2887        1.7000 164,064.9471            0.1291
  wls_elogit 26     0.9987    0.1359      0.0951       1.2808        1.6868 160,203.7717            0.1288
   bin_logit 25     1.0002    0.1361      0.1010       1.2824        1.6882 160,531.6393            0.1346
quasi_spline 29     1.0122    0.1357      0.1056       1.2978        1.7028 163,104.8306            0.1313
  ols_elogit 26     1.0304    0.1471      0.3588       1.3151        1.7112 164,879.2255            0.1577
     betareg 26     1.0336    0.1514      0.3577       1.3041        1.7059 163,903.4835            0.1554
  bb_selecao 19     1.0348    0.1558      0.3868       1.3037        1.6933 161,254.8255            0.0755
    bb_logit 26     1.0363    0.1521      0.3561       1.3080        1.7057 163,836.4470            0.165

**Aplicação da regra.** Considera-se empatado com o melhor todo modelo cujo erro médio por UF fica dentro de 1 EP do melhor; escolhe-se o de menos parâmetros (desempate: MAE ponderado no 5-fold).
O vice é o melhor modelo por erro médio quando ele não é o escolhido; caso contrário, o segundo colocado.

In [4]:
ordem = TL.sort_values("mae_uf_pp").reset_index(drop=True)
melhor, se = ordem.loc[0, "mae_uf_pp"], ordem.loc[0, "se_uf_pp"]
emp = ordem[ordem.mae_uf_pp <= melhor + se].merge(T5[["modelo", "mae_pond_pp"]], on="modelo", suffixes=("", "_5f"))
ESC = emp.sort_values(["k", "mae_pond_pp_5f" if "mae_pond_pp_5f" in emp else "mae_pond_pp"]).iloc[0]["modelo"]
VICE = ordem.loc[0, "modelo"] if ordem.loc[0, "modelo"] != ESC else ordem.loc[1, "modelo"]
print(f"melhor por erro médio: {ordem.loc[0,'modelo']} = {melhor:.3f} p.p. (EP {se:.3f}); limite 1-EP = {melhor+se:.3f}")
print("empatados (dentro de 1 EP):", list(emp.modelo))
print("ESCOLHIDO:", ESC, "| VICE:", VICE)
fig = go.Figure(go.Scatter(x=ordem.mae_uf_pp, y=ordem.modelo, mode="markers", error_x=dict(array=ordem.se_uf_pp), marker=dict(size=8, color="#1b6ca8")))
fig.add_vline(x=melhor + se, line_dash="dash", line_color="#888", annotation_text="melhor + 1 EP")
fig.update_layout(title="LOSO: erro médio do share da UF (p.p.), barras = ±1 EP", xaxis_title="p.p.", yaxis=dict(autorange="reversed"), height=520, template="plotly_white")
fig.write_html(FIG / "cv_loso_mae_uf.html", include_plotlyjs="cdn", div_id="cv_loso_mae_uf")
o5 = T5.sort_values("mae_pond_pp")
fig = go.Figure(go.Scatter(x=o5.mae_pond_pp, y=o5.modelo, mode="markers", error_x=dict(array=o5.se_mae_pond_pp), marker=dict(size=8, color="#c2571a")))
fig.update_layout(title="5-fold: MAE municipal ponderado por votos (p.p.), ±1 EP dos folds", xaxis_title="p.p.", yaxis=dict(autorange="reversed"), height=520, template="plotly_white")
fig.write_html(FIG / "cv_kfold_mae.html", include_plotlyjs="cdn", div_id="cv_kfold_mae")

melhor por erro médio: bin_probit = 0.998 p.p. (EP 0.135); limite 1-EP = 1.133
empatados (dentro de 1 EP): ['bin_probit', 'wls_elogit', 'bin_logit', 'quasi_spline', 'ols_elogit', 'betareg', 'bb_selecao', 'bb_logit', 'quasi_reglag', 'bin_cauchit', 'bb_spline', 'glmm_uf', 'bin_loglog']
ESCOLHIDO: bb_selecao | VICE: bin_probit


### Seleção aninhada: a regra de 1 EP com o `bb_selecao` sem vazamento de seleção

**Problema.** A escolha das 11 contínuas e a fusão Nordeste + Norte do `bb_selecao` (notebook 02, Parte 8) usaram o 2º turno de 2026 de **todas** as UFs. O LOSO acima deixa uma UF fora do ajuste dos coeficientes, mas não da seleção, então o erro do `bb_selecao` é otimista.

**Regra (fixada antes de rodar).** Em cada uma das 27 dobras LOSO, só com as 26 UFs de treino: (1) testa-se a fusão Nordeste + Norte e a de blocos de prefeito pela razão de verossimilhanças (funde se p > 0,05) no modelo cheio; (2) o passo a passo por BIC (backward, forward e both) fica com o conjunto de menor BIC; (3) ajusta-se o modelo e prevê-se a UF retida com n̂. O MAE por UF dessas 27 previsões (`results/cv/loso_bb_selecao_aninhada.parquet`, produzido por `src/selecao.py`) **substitui** o do `bb_selecao` na regra de 1 EP e no limite de aceitação L do notebook 05. Os demais modelos da tabela não têm seleção de variáveis: ficam como estão. As 27 dobras não são refeitas aqui.

In [5]:
from src import selecao as SEL
ANINH = pd.read_parquet(CV / "loso_bb_selecao_aninhada.parquet").set_index("uf").loc[UFS_]
assert len(ANINH) == len(UFS_) and ANINH.erro_abs_pp.notna().all()
mae_ani = SEL.mae_uf_pp(ANINH.reset_index())
se_ani = float(ANINH.erro_abs_pp.std(ddof=1) / np.sqrt(len(ANINH)))
tl = TL.set_index("modelo")
mae_comp, se_comp = float(tl.loc["bb_selecao", "mae_uf_pp"]), float(tl.loc["bb_selecao", "se_uf_pp"])
ref = set(sel["variaveis"]) - {"lag"}
u_comp = TUF[TUF.modelo == "bb_selecao"].set_index("uf").loc[UFS_]
POR_UF = pd.DataFrame({
    "n_vars": ANINH.n_vars, "fusao_regiao": ANINH.fusao_regiao.replace("", "nenhuma"), "fusao_bloco": ANINH.fusao_bloco,
    "metodo": ANINH.metodo, "erro_aninhado_pp": ANINH.erro_pp, "erro_completo_pp": u_comp.erro_uf_pp,
    "a_mais_que_registro": ANINH.variaveis.map(lambda s: ",".join(sorted(set(s.split(",")) - ref))),
    "a_menos_que_registro": ANINH.variaveis.map(lambda s: ",".join(sorted(ref - set(s.split(","))))),
})
print("Antes e depois (MAE por UF do bb_selecao no LOSO, p.p.)")
print(pd.DataFrame({"seleção em todas as UFs (registro)": [mae_comp, se_comp], "seleção dentro de cada dobra (aninhada)": [mae_ani, se_ani]},
                   index=["MAE por UF", "EP"]).to_string())
print(f"\nDiferença: {mae_ani - mae_comp:+.4f} p.p. ({(mae_ani - mae_comp) / se_ani:.2f} EP aninhados)")
print("\nPor UF"); print(POR_UF.to_string())
print("\nNº de contínuas por dobra:", ANINH.n_vars.value_counts().sort_index().to_dict(), "| mediana", ANINH.n_vars.median(), "| registro:", len(ref))
print("Fusão de região:", ANINH.fusao_regiao.replace("", "nenhuma").value_counts().to_dict(), "| fusão de bloco (vazio = todos na referência):", ANINH.fusao_bloco.value_counts().to_dict(), "| método:", ANINH.metodo.value_counts().to_dict())
print("Dobras com exatamente o conjunto do registro:", int(sum(set(s.split(",")) == ref for s in ANINH.variaveis)), "de", len(ANINH))
FREQ = pd.Series([v for s in ANINH.variaveis for v in s.split(",")]).value_counts()
print("Variáveis por nº de dobras em que entram:", FREQ.to_dict())
print("Maiores erros absolutos:", ANINH.erro_abs_pp.nlargest(3).round(2).to_dict())

# regra de 1 EP com o valor aninhado no lugar do bb_selecao
ordem2 = TL.sort_values("mae_uf_pp").reset_index(drop=True)
mk = ordem2.modelo == "bb_selecao"
ordem2.loc[mk, ["mae_uf_pp", "se_uf_pp"]] = [mae_ani, se_ani]
ordem2 = ordem2.sort_values("mae_uf_pp").reset_index(drop=True)
melhor2, se2 = ordem2.loc[0, "mae_uf_pp"], ordem2.loc[0, "se_uf_pp"]
emp2 = ordem2[ordem2.mae_uf_pp <= melhor2 + se2].merge(T5[["modelo", "mae_pond_pp"]], on="modelo", suffixes=("", "_5f"))
ESC2 = emp2.sort_values(["k", "mae_pond_pp_5f"]).iloc[0]["modelo"]
print(f"\nmelhor: {ordem2.loc[0, 'modelo']} = {melhor2:.4f} p.p. (EP {se2:.4f}); limite 1-EP = {melhor2 + se2:.4f}")
print("bb_selecao aninhado:", f"{mae_ani:.4f}", "dentro do limite:", bool(mae_ani <= melhor2 + se2), "| k =", int(ordem2.loc[ordem2.modelo == "bb_selecao", "k"].iloc[0]))
print("empatados:", list(emp2.modelo)); print("ESCOLHIDO:", ESC2, "(sem aninhar:", ESC, ")")
assert ESC2 == ESC, "a seleção aninhada mudou a escolha: revisar o notebook 05 e os documentos"
# contrato com o notebook 05: o MAE aninhado do bb_selecao define o limite L de aceitação das extensões
pd.DataFrame([{"modelo": "bb_selecao", "mae_uf_pp_completo": mae_comp, "se_uf_pp_completo": se_comp,
               "mae_uf_pp_aninhado": mae_ani, "se_uf_pp_aninhado": se_ani, "n_ufs": len(ANINH),
               "melhor_modelo": ordem2.loc[0, "modelo"], "se_melhor_loso": float(se2),
               "limite_1ep_aninhado": float(melhor2 + se2), "escolhido_aninhado": ESC2}]).to_csv(TAB / "cv_loso_aninhado.csv", index=False)

Antes e depois (MAE por UF do bb_selecao no LOSO, p.p.)
            seleção em todas as UFs (registro)  seleção dentro de cada dobra (aninhada)
MAE por UF                              1.0348                                   1.1168
EP                                      0.1558                                   0.1617

Diferença: +0.0820 p.p. (0.51 EP aninhados)

Por UF
    n_vars                         fusao_regiao                   fusao_bloco    metodo  erro_aninhado_pp  erro_completo_pp                        a_mais_que_registro            a_menos_que_registro
uf                                                                                                                                                                                                    
AC      11                       Nordeste+Norte  esquerda;centro+sem_prefeito   forward           -2.0512           -2.0512                                                                           
AL      13               Norde

**Leitura (seleção aninhada).** Repetir a seleção dentro de cada dobra sobe o MAE por UF do `bb_selecao` de 1,0348 p.p. (EP 0,156) para 1,1168 p.p. (EP 0,162), 0,082 p.p. a mais, cerca de metade de um EP (0,51): o registro era otimista, mas pouco. O limite de 1 EP continua sendo o do melhor modelo (`bin_probit`, 0,998 + 0,135 = 1,133 p.p.), e 1,117 fica dentro dele, com folga de 0,016 p.p. `bb_selecao` segue empatado e é o de menor k (19), então **a escolha não muda**. Estabilidade da seleção por dobra: 10 a 13 contínuas (mediana 11, contra 11 no registro); Nordeste + Norte fundidos em 17 das 27 dobras e, em outras 7, também com o Sudeste (a região de referência); Centro-Oeste + Norte com Nordeste + Sudeste em 1 (GO retida), Norte + Sudeste em 1 (MA) e nenhuma fusão em 1 (BA). Blocos de prefeito: esquerda e centro + sem prefeito fundidos em 16 dobras, só esquerda em 10 (centro e sem prefeito na referência direita) e esquerda e sem prefeito em 1. Só 9 dobras reproduzem exatamente o conjunto do registro; seis variáveis entram em todas as dobras (`pct_dep_esq`, `comparecimento`, `log_dens`, `pct_mulheres`, `cobertura_pbf`, `pct_vab_agro`) e `pct_analfabeto` (21 dobras), `pct_superior` (19), `delta_comp` (11), `log_pop` e `pct_urbana` (8 cada) entram em algumas. Os maiores erros são AP (-3,17), DF (-2,59) e GO (+2,45). Valor para o notebook 05: o limite de aceitação passa a usar 1,1168 no lugar de 1,0348 (`results/tables/cv_loso_aninhado.csv`).

In [6]:
def calib(pi, q=20):
    d = pd.DataFrame({"pi": pi, "y": y, "n": n}); d["b"] = pd.qcut(d.pi, q, duplicates="drop")
    g = d.assign(v=d.pi * d.n).groupby("b", observed=True)[["v", "y", "n"]].sum()
    return g.v / g.n, g.y / g.n
outro = [k for k in ordem.modelo if k not in (ESC, VICE)]
mostrar = [ESC, VICE] + [outro[-1]]   # o pior da lista como contraste
fig = go.Figure(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color="#999", dash="dash"), name="calibração perfeita"))
for k, c in zip(mostrar, ["#1b6ca8", "#c2571a", "#4c9a2a"]):
    p, o = calib(PIL[k]); fig.add_trace(go.Scatter(x=p, y=o, mode="lines+markers", name=k, line=dict(color=c)))
fig.update_layout(title="Calibração LOSO: π̂ médio (ponderado por votos) vs share observado, 20 faixas de π̂", xaxis_title="π̂ previsto", yaxis_title="share observado",
                  template="plotly_white", height=520)
fig.write_html(FIG / "cv_calibracao.html", include_plotlyjs="cdn", div_id="cv_calibracao")
for k in mostrar:
    p, o = calib(PIL[k]); print(k, "| máx |previsto - observado| nas faixas (p.p.):", round(float((p - o).abs().max() * 100), 2))

bb_selecao | máx |previsto - observado| nas faixas (p.p.): 0.63
bin_probit | máx |previsto - observado| nas faixas (p.p.): 0.9
l1_logit | máx |previsto - observado| nas faixas (p.p.): 3.33


### Sensibilidade das estimativas regionais ao modelo

O erro LOSO **regional** não é uma métrica justa para escolher modelo: num binomial de ligação canônica (logit) com dummies de região, a equação de escore de cada dummy
força Σ_u w_u δ_u = 0 (δ_u = resíduo de share da UF u dentro da amostra, w_u = seu peso em votos na região). Ao reter a UF u, o intercepto da região se move em
Δ = −w_u δ_u / (1 − w_u), e o erro retido da UF fica ≈ δ_u / (1 − w_u). O erro regional LOSO é Σ_u w_u δ_u / (1 − w_u): com pesos iguais vale
(1/(1 − w)) Σ_u w δ_u = 0 exatamente; com pesos desiguais fica perto de zero (a aproximação ignora a mudança nos demais coeficientes). O probit satisfaz as equações
ponderadas só aproximadamente. O beta-binomial pesa cada município por n/(1 + (n−1)ρ), então seu escore não é ponderado por votos e o cancelamento não ocorre.
Um erro regional LOSO pequeno num binomial canônico revela esse mecanismo, não um modelo melhor. A métrica de seleção justa é o erro por UF.

A tabela abaixo compara a estimativa pontual LOSO de `bin_probit`, `bin_logit` e `bb_selecao`. Também mostra o nº de UFs
e o maior peso w_u por região, que indica onde o cancelamento não é exato (SP no Sudeste).

In [7]:
MODS = ["bin_probit", "bin_logit", "bb_selecao"]
obs_r = obs_reg.copy(); obs_r["Brasil"] = y.sum() / n.sum()
w_uf = pd.Series(nh).groupby([reg, uf]).sum()
w_uf = w_uf / w_uf.groupby(level=0).sum()
SENS = pd.DataFrame({"real_pct": obs_r * 100})
for m in MODS:
    est = M.agregar(PIL[m], nh, reg) * 100
    SENS[f"{m}_pct"] = est; SENS[f"{m}_erro_pp"] = est - SENS["real_pct"]
SENS = SENS.loc[sorted(set(SENS.index) - {"Brasil"}) + ["Brasil"]]
SENS.insert(1, "n_ufs", [int(pd.Series(uf)[reg == r].nunique()) if r != "Brasil" else int(pd.Series(uf).nunique()) for r in SENS.index])
wm = [w_uf[r].max() if r != "Brasil" else (pd.Series(nh).groupby(uf).sum() / nh.sum()).max() for r in SENS.index]
SENS.insert(2, "w_max_uf", wm)
SENS.index.name = "regiao"
SENS.reset_index().to_csv(TAB / "sensibilidade_regional.csv", index=False)
print(SENS.round(3).to_string())
cv = pd.read_csv(TAB / "cv_loso.csv").set_index("modelo").loc[MODS]
print(cv[["mae_uf_pp", "mae_reg_pp", "max_reg_pp"]].round(3).to_string())
rg = SENS.drop("Brasil")
est_cols = [f"{m}_pct" for m in MODS]
print("amplitude entre modelos (máx - mín) por região, p.p.:", (rg[est_cols].max(axis=1) - rg[est_cols].min(axis=1)).round(3).to_dict())
print("Brasil:", round(float(SENS.loc["Brasil", est_cols].max() - SENS.loc["Brasil", est_cols].min()), 3))

              real_pct  n_ufs  w_max_uf  bin_probit_pct  bin_probit_erro_pp  bin_logit_pct  bin_logit_erro_pp  bb_selecao_pct  bb_selecao_erro_pp
regiao                                                                                                                                           
Centro-Oeste   39.9910      4    0.4240         40.0370              0.0460        40.0260             0.0340         40.3800              0.3890
Nordeste       69.3690      9    0.2540         69.4160              0.0480        69.4240             0.0550         69.3550             -0.0130
Norte          50.0050      7    0.4830         50.1350              0.1300        50.1570             0.1520         49.4420             -0.5630
Sudeste        47.2420      4    0.5110         47.4720              0.2300        47.4780             0.2360         47.6460              0.4040
Sul            37.4240      3    0.3780         37.3940             -0.0300        37.3970            -0.0270         36.859

### Leitura

**Seleção (regra de 1 EP).** O melhor modelo por erro médio de UF no LOSO é o `bin_probit`: 0,998 p.p. (EP 0,135). O limite de empate é 0,998 + 0,135 = 1,133 p.p., e 13 dos 18 modelos avaliados ficam dentro dele. Pela regra fixada antes dos resultados, vence o de menos parâmetros entre os empatados: o beta-binomial `bb_selecao`, com k = 19 contra 25 do `bin_probit`. Seu erro é 1,035 p.p. (EP 0,156), uns 0,04 p.p. acima do melhor, diferença bem menor que 1 EP. Em termos de voto: o modelo escolhido erra o share de um estado inédito em cerca de 1 p.p., e os modelos empatados não se distinguem estatisticamente nessa métrica. O vice é o `bin_probit`.

**O que mudou com a correção do vazamento.** Com covariáveis anteriores ao 2º turno e n̂ no município retido, o erro LOSO subiu: o melhor passou de 0,930 p.p. (execução anterior, com vazamento) para 0,998 p.p. A decisão não mudou: o beta-binomial continua escolhido, agora como `bb_selecao` (12 termos: 11 contínuas e a defasagem; Nordeste e Norte fundidos na média, teste da razão de verossimilhanças p = 0,921, notebook 02, Parte 8). O erro mais alto é o mais honesto, porque o modelo não usa mais informação que não existiria na véspera da apuração.

**Seleção aninhada e limite do notebook 05.** O 1,035 acima ainda usa a seleção feita com todas as UFs. Com a seleção refeita em cada dobra o `bb_selecao` erra 1,117 p.p. (EP 0,162), a escolha se mantém e o limite que alimenta o notebook 05 passa a ser L = 1,1168 + 0,1347 = 1,2515 p.p. (1,1695 com o valor sem aninhar). As extensões testadas no notebook 05 (`bb_disp`, `bb_slx`) ainda usam a seleção dos dados completos; contra um L aninhado o limite fica, portanto, mais folgado do que seria para uma comparação em pé de igualdade, e nenhuma decisão está perto dele.

**Calibração.** Entre as 20 faixas de π̂ no LOSO, a maior distância entre share previsto e observado é 0,63 p.p. no `bb_selecao` e 0,90 p.p. no `bin_probit`, contra 3,33 p.p. no `l1_logit` (pior da lista). O escolhido não superestima nem subestima sistematicamente nenhuma faixa de voto lulista.

**Sensibilidade regional.** O erro regional LOSO é pequeno no binomial canônico por construção (`bin_logit`: no máximo 0,236 p.p.; `bin_probit`: 0,230 p.p.), efeito do cancelamento descrito acima e não prova de modelo melhor. O `bb_selecao` não tem esse cancelamento e mostra o erro regional real de um modelo sem as UFs retidas: Centro-Oeste +0,389, Nordeste -0,013, Norte -0,563, Sudeste +0,404, Sul -0,564 p.p. e Brasil -0,106 p.p. Portanto, as estimativas regionais dependem do modelo em meio ponto percentual no Norte (amplitude entre os três modelos 0,715 p.p.) e no Sul (0,537 p.p.), e em menos de 0,2 p.p. no Nordeste (0,068) e no Sudeste (0,174). No total nacional os três concordam em 0,061 p.p. Uma diferença regional menor que cerca de 0,6 p.p. entre Lula e Flávio não deve ser lida como resultado firme.